# A-1/A-3/A-10 — 특징별 AUC(95% CI) · Ablation · σ 스윕 · 가중 앙상블


In [ ]:
import os, sys, subprocess, pickle, io as _io
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
from torchvision.models import ResNet50_Weights
from torchvision.transforms.functional import gaussian_blur
from PIL import Image as _Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

SEARCH_ROOTS=['/home','/root','/workspace',os.path.expanduser('~'),'.','..','../..','.']
def _find(name, ftype='f', maxdepth=8):
    res=[]
    for root in SEARCH_ROOTS:
        if not os.path.exists(root): continue
        try:
            out=subprocess.run(['find',root,'-maxdepth',str(maxdepth),'-type',ftype,'-name',name],
                               capture_output=True,text=True,timeout=20).stdout.strip()
            if out: res+=[p for p in out.split('\n') if p]
        except: pass
    return sorted(set(res))

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED=42; np.random.seed(SEED); torch.manual_seed(SEED)

CIFAR_MEAN=[0.4914,0.4822,0.4465]; CIFAR_STD=[0.2470,0.2435,0.2616]
IMGNET_MEAN=[0.485,0.456,0.406];   IMGNET_STD=[0.229,0.224,0.225]
def make_preprocess(ds):
    m,s=(CIFAR_MEAN,CIFAR_STD) if 'CIFAR' in ds else (IMGNET_MEAN,IMGNET_STD)
    mean=torch.tensor(m).view(1,3,1,1); std=torch.tensor(s).view(1,3,1,1)
    return lambda x:(x/255.0-mean.to(x.device))/std.to(x.device)

def load_backbone(ds):
    if ds=='CIFAR-10':
        ck=(_find('resnet50_cifar10_finetuned.pt') or [None])[0]
        m=models.resnet50(weights=None); m.fc=nn.Linear(2048,10)
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); nc=10
    elif ds=='CIFAR-100':
        ck=(_find('resnet50_cifar100_finetuned.pt') or [None])[0]
        m=models.resnet50(weights=None); m.fc=nn.Linear(2048,100)
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); nc=100
    elif ds=='SVHN':
        ck=(_find('resnet50_svhn_finetuned.pt') or [None])[0]
        m=models.resnet50(weights=None); m.fc=nn.Linear(2048,10)
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); nc=10
    elif ds in ('TinyImageNet','Tiny ImageNet'):
        ck=(_find('resnet50_tinyimagenet_finetuned.pt') or [None])[0]
        m=models.resnet50(weights=None); m.fc=nn.Linear(2048,200)
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); nc=200
    else:  # ImageNet
        m=models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2); nc=1000
    return m.to(device).eval(), nc

# mixed_dataset.pkl finder
def find_mixed():
    out={}
    for p in _find('mixed_dataset.pkl'):
        pl=p.lower()
        if 'cifar10' in pl or 'cifar_10' in pl: out.setdefault('CIFAR-10',p)
        elif 'cifar100' in pl: out.setdefault('CIFAR-100',p)
        elif 'svhn' in pl: out.setdefault('SVHN',p)
        elif 'tiny' in pl: out.setdefault('TinyImageNet',p)
        elif 'imagenet' in pl and 'eps8' in pl: out.setdefault('ImageNet_eps8',p)
    return out

def gb(x,sigma):
    k=int(2*np.ceil(3*sigma)+1); k=k+1 if k%2==0 else k
    return gaussian_blur(x,kernel_size=k,sigma=sigma)
def to224(img,mode='bicubic'):
    if img.dim()==3: img=img.unsqueeze(0)
    img=img.float()
    if img.shape[-1]!=224 or img.shape[-2]!=224:
        img=F.interpolate(img,size=(224,224),mode=mode,align_corners=False)
    return img.clamp(0,255)
def jpeg(img224,q):
    arr=img224.detach().squeeze(0).permute(1,2,0).clamp(0,255).byte().cpu().numpy()
    buf=_io.BytesIO(); _Image.fromarray(arr).save(buf,format='JPEG',quality=int(q)); buf.seek(0)
    return torch.from_numpy(np.array(_Image.open(buf).convert('RGB'))).float().permute(2,0,1).unsqueeze(0).to(img224.device)
def median3(img224):
    x=F.pad(img224,(1,1,1,1),mode='reflect')
    p=x.unfold(2,3,1).unfold(3,3,1)
    return p.contiguous().view(*p.shape[:4],9).median(dim=-1).values

def load_mixed(pkl_path, n_clean=500):
    with open(pkl_path,'rb') as f: mixed=pickle.load(f)
    clean=[im for (im,lb,atk) in mixed if atk=='clean']
    adv  =[(im,atk) for (im,lb,atk) in mixed if atk!='clean']
    rng=np.random.RandomState(SEED); idx=np.arange(len(clean)); rng.shuffle(idx)
    clean=[clean[i] for i in idx[:n_clean]]
    return clean, adv

def auc_ci(neg,pos,n_boot=1000,seed=SEED):
    y=np.r_[np.zeros(len(neg)),np.ones(len(pos))]; s=np.r_[neg,pos]
    if len(set(y))<2: return float('nan'),float('nan'),float('nan')
    base=roc_auc_score(y,s); rng=np.random.RandomState(seed); b=[]; N=len(y)
    for _ in range(n_boot):
        ii=rng.randint(0,N,N)
        if len(set(y[ii]))>1: b.append(roc_auc_score(y[ii],s[ii]))
    lo,hi=(np.percentile(b,[2.5,97.5]) if b else (float('nan'),float('nan')))
    return base,lo,hi
print('header ready; device=',device)

In [ ]:
# blur_edge 캐시 적재
DS_KEYS=['CIFAR-10','CIFAR-100','SVHN','TinyImageNet','ImageNet_eps8']
CACHES={}
for ds in DS_KEYS:
    c=[p for p in _find(f'features_blur_edge_{ds}.pkl') if 'blur_edge_results' in p]
    if c: CACHES[ds]=c[0]
FD={}
for ds,p in CACHES.items():
    with open(p,'rb') as f: FD[ds]=pickle.load(f)
    print(f"✓ {ds}: feats={[k for k in FD[ds] if k not in ('labels','attacks')]}")
print('loaded',list(FD.keys()))

In [ ]:
# 캘리브레이션 분리 z-score + |z| anomaly
def zfit(arr,labels):
    a=np.array(arr,float); lab=np.array(labels)
    ci=np.where(lab==0)[0]; rng=np.random.RandomState(SEED); rng.shuffle(ci)
    half=ci[:len(ci)//2]
    mu=a[half].mean(); sd=a[half].std()+1e-8
    return (a-mu)/sd, set(ci[len(ci)//2:].tolist())
def feat_auc(fd,key,atk=None,n_boot=500):
    lab=np.array(fd['labels']); atks=np.array(fd['attacks'])
    z,test_clean=zfit(fd[key],lab); anom=np.abs(z)
    keep=np.array([ (i in test_clean) or (lab[i]==1) for i in range(len(lab)) ])
    if atk: keep=keep & ((atks=='clean')|(atks==atk))
    neg=anom[keep & (lab==0)]; pos=anom[keep & (lab==1)]
    return auc_ci(neg,pos,n_boot=n_boot)
KEYS=[('HF-Energy σ=0.5','hf_energy_s0p5'),('HF-Energy σ=1.0','hf_energy_s1p0'),
      ('GaussianL1 σ=0.5','gauss_l1_s0p5'),('GaussianL1 σ=1.0','gauss_l1_s1p0'),
      ('GaussianL1 σ=2.0','gauss_l1_s2p0'),('PredL1-JPEG','predl1_jpeg'),
      ('EdgePres-L1','edge_pres_l1'),('Sobel','sobel_energy')]
for ds,fd in FD.items():
    atks=sorted(set(a for a in fd['attacks'] if a!='clean'))
    print(f"\n[{ds}] 특징별 AUROC (95% CI)")
    print(f"  {'Feature':<20}"+''.join(f'{a:>22}' for a in atks)+f"{'Overall':>22}")
    for nm,k in KEYS:
        if k not in fd: continue
        row=f"  {nm:<20}"
        for a in atks+[None]:
            b,lo,hi=feat_auc(fd,k,a)
            row+=f'{b:>9.4f}[{lo:.3f},{hi:.3f}]'.rjust(22)
        print(row)

In [ ]:
# 앙상블 ablation
def ens_anom(fd,keys):
    lab=np.array(fd['labels']); zs=[]
    for k in keys:
        z,_=zfit(fd[k],lab); zs.append(z)
    ens=np.mean(zs,axis=0)
    ci=np.where(lab==0)[0]; mu=ens[ci].mean()
    return np.abs(ens-mu)
def ens_auc(fd,keys,atk=None):
    lab=np.array(fd['labels']); atks=np.array(fd['attacks']); anom=ens_anom(fd,keys)
    keep=np.ones(len(lab),bool)
    if atk: keep=(atks=='clean')|(atks==atk)
    try: return roc_auc_score(lab[keep],anom[keep])
    except: return float('nan')
BASE={'CIFAR-10':['hf_energy_s0p5'],'CIFAR-100':['hf_energy_s0p5','gauss_l1_s1p0','predl1_jpeg'],
      'SVHN':['hf_energy_s0p5'],'TinyImageNet':['hf_energy_s0p5','gauss_l1_s1p0','predl1_jpeg'],
      'ImageNet_eps8':['hf_energy_s0p5','gauss_l1_s1p0','predl1_jpeg']}
ALL=['hf_energy_s0p5','gauss_l1_s1p0','predl1_jpeg']
for ds,fd in FD.items():
    base=[k for k in BASE.get(ds,ALL) if k in fd]
    full=ens_auc(fd,base); print(f"\n[{ds}] 최적후보={base}  Full Overall AUC={full:.4f}")
    print("  Δ_remove (제거 시 변화):")
    for k in base:
        sub=[x for x in base if x!=k]
        if sub: print(f"    -{k:<18} {ens_auc(fd,sub)-full:+.4f}")
    print("  Δ_add (추가 시 변화):")
    for k in ALL+['gauss_l1_s0p5','hf_energy_s1p0']:
        if k in fd and k not in base:
            print(f"    +{k:<18} {ens_auc(fd,base+[k])-full:+.4f}")

In [ ]:
# σ 스윕 + 단순평균 vs 정수가중치 앙상블
import itertools
print("σ 스윕 (HF-Energy Overall AUC):")
for ds,fd in FD.items():
    row=f"  {ds:<16}"
    for k in ['hf_energy_s0p5','hf_energy_s1p0']:
        if k in fd: row+=f" {k}={ens_auc(fd,[k]):.4f}"
    print(row)
print("\n가중 앙상블 탐색 (HF:Gauss-1.0:PredL1, 정수 1..4):")
for ds,fd in FD.items():
    keys=[k for k in ['hf_energy_s0p5','gauss_l1_s1p0','predl1_jpeg'] if k in fd]
    if len(keys)<3: continue
    lab=np.array(fd['labels']); zs=[zfit(fd[k],lab)[0] for k in keys]
    best=(0,(1,1,1))
    for w in itertools.product(range(1,5),repeat=3):
        ens=sum(wi*z for wi,z in zip(w,zs))/sum(w)
        ci=np.where(lab==0)[0]; anom=np.abs(ens-ens[ci].mean())
        try: a=roc_auc_score(lab,anom)
        except: a=0
        if a>best[0]: best=(a,w)
    eq=ens_auc(fd,keys)
    print(f"  {ds:<16} 균등={eq:.4f}  최적가중{best[1]}={best[0]:.4f}  Δ={best[0]-eq:+.4f}")